# R05 — Independent grading and semantic adjudication

Runs on CPU. Uses previously completed R02 and R03 records, with NO generation or invented labels. Creates 144 pseudonymized response rows (24 shared problems × 2 models × 3 phases), two reviewer CSVs, and an unshared sealed key. The review sample is selected by deterministic case hash, not by model accuracy.

In [ ]:
import base64,zlib,json,hashlib,csv,os,math,statistics,zipfile
from pathlib import Path
import pandas as pd
from scipy.stats import binomtest
OUT=Path('/content/RSX_R05_Manual_Rubric')
OUT.mkdir(parents=True,exist_ok=True)

def digest(x):return hashlib.sha256(json.dumps(x,sort_keys=True,ensure_ascii=False,separators=(',',':')).encode()).hexdigest()
def unpack(blob):return json.loads(zlib.decompress(base64.b64decode(blob)))
MANIFEST=unpack('eNq9feuP5DZ2778iDMbwl562SL395WJsZ6/tXWed9eAmSBAElEh1ya2SKpKqa3oX+d9zHqReJc0oE+zFeu2WxOfh4Tm/8yDrb28K1Zv+zbfev/3tjVYDPAzw8OapP6fPbx68N5UeH78N/SJIQqMyP4n9spT4ve2qp6pR9X9UjTYfoWySwdtL154v1NDP7anxlG4vQ+8pT7dPj573o/EG9Wx6bzgZfOUNLf351LXt2XQP3u1UFSevaHuo9Fb4PtT5MH332rI0Xe+dqjM0Gfhfebrqi/baDF7Zdl5uquYJPjTm5hXXfsAq2Gl7885XaFa3tmdqDstiR/8HJzOo7onmn/iP/n89eJ8niV/qKDeFCpQOYl2KTZIIP40XRPlFdYWpve7aIFHyqngtauPBUDvz6P1Y9d5ZVY0H5fW1QLp1QLBTZ+DfrxcYfFtCHaDft94vH7578L775V8ePNVo70Nnnp9hQo9ErUtXFQbLto3Bgh60+zbyfaqADydVl0SIsaRquEVsTI3NUWdUO4xgKX5qqMVz2wynB89OpW9rrDK0g6qxpcD3eYyegoJP1A0Pvq76wWiYJvbe0gDO3s3AFBf99TwKEX3FH3HM9J4n15l+sLW5b+6MiuJcsa/H2ZpX2o3UqK7xSlgKrzd1zd0R/XtPX7uKxqoGnt6SKUQgZHSUMcrcVzJUsdGJzLI03GSMNBQLvvi/Ku8qGONNNbDssCvy6ytQtVAdcTZuBaArr87lhKtAr1PcIa7uSfEy+zzJW9sRTdsGiGOeTaN7auAE65l3LdCv857Ui6Hd9FZCNej2ZOoLvphREOhhXnnvIMmHqq5hhxm9JFEU+0cJlEbCxGGQlWFoosiY7Z2TLujzfa0qWOAzCQ/Y+p55gpmdTW0Gz7yY7hWG2TU4XRIEnVHPpeoHt/tVgzP4q2loC0FdYBicRw8zMrDmsOdColK/kgZHhUEYR1KVUgR5Alsl3pxSFC9lwW+nqm5PuL3C0HsF7oTBAT8PrVavMHDYbQm/fkDC03jzURxUZyQE11A9LWpjLidzs1PG99X8Pbe74uujSyaDUpd+WpgU5L80ent+/mJ6f6rbV01MCcIFl6tU3ZmFXAFS/hkXgwWd8WQkeVUuwJQwTGZUQxu1Z0FB/C65AK7k40quc3fIIMjxY2O4qMtJyyw8OGutTRiHaSITWWo/3BHxUi6X9b2Xq74qkOmq4trhllEdyvLBextEqOFA4sMuNvrucwi7+b1XmG5ALdCrGmZiVZ4EZQd/elXpvbZXD6bi5bCHH73frpdL2xvvj6ArGqBsCw8kQXRLJYiUt/aKfFL3LUhhoBB8Vlp7japq0DEDUUwBJWFzoFgoYSMZ5LuFOg6IoFhnRntuGffRBZYNRq3qeknv7CiTFUDsKCrjWJsgkIW/Se4gXXIZTBvYK2+7rr0ZDYN8iCOfmKWsXqyuIuJCqaoZDKmPTg2k9YT/FRAQxo5cCjS5EOd55j+voMtAfSGwcNKlQWIvWa6nmi/GVUXybGiP1A8OUkAERoS+ytNYK6WyaHubhQsCoE78ob3mQ4kCskcd2zFw0F31ghLxBHISxAewQuI9V8OAG49wGdCLNAWq0rotcM5NdVYkFusBN88JWAq+VjgvUhhQqCZSVQ3Q+Mxbk9QmFxtuoEPaK/BjoYaeqfM7QDH4S7vOmeS9eu255q9qKE6vDzQMmAQUhroPtCZYCwQeghT82lzPOaHAcfy2yQe7rT501cfKcFNuwLOmhHyc9IEjxYgKJyqW6lzVr14DRXEGy+UMDwvNxI/DrAhiAxLTFNviI06zFUBs2hp37vXpNDAP1y2glUt7YfVdtlecVaMrY+mH2xMY/1EiRCt5I7s6bu/6jyHsm9O4a1GZCH/VNLxwDVObgAO+TB2GflCIUCcmkHGqgm0IJES25OQ/IC/imvVFZRpY8qJWPeyy4aqBO77rlH71agRxRvPEaaY3ALMGBQ8BQqNosUMfoE0N4pMhc922mrmOReC8EQXcrrpnFgFL3Uqkosr4pLirR+97VRfXGkXIkifXPdp9AVuIeoCZ3KozmR2L0b6Yx6VSloeRlB9keZSGSuRFnqgy2zbLgjXqMLVaMBiA45pRZW/RJohQIi4+Adp6R1INhg26qq7PphlQT8mvJonq0ElxwmloqkgtXJtqmKsLLGeHgCKTynDrWCWwAnulsaPoKOMlSVGoMsJ/SV/ubLkVDvuLem2YbWIfYSRapnWNcAwWsAHF2vcVciBuEftApYNvwhWvMFSGp/eg2OtK0X58T38yOoe1ne9C0p1EAhKUFVk1gFRNUWnkT9bSQwtkwH3hJDF10mhiK8PaCqTVaG6tNi30eRjvxEUpwzTMtExEvA3MMxmu4A7tmndDCyPua0Qc5/YFiEJ6E17caku7tqlxwUEckV45kWV5veBuQJg6QCsAXv9ALHc2bE3C1gLbBuR7Ncw8B9xNiJqmI21EsI8sza66wPLdGvtkQBlcAGxiP2NbOJiirs45IDX+BHzczgfxvu+vtFMR10B12606z/e7bW8ET9g3KiK0J7gCCBrYG/c1Tqgem3FgiBtmAx9dIlcYZ1US+HdmYQGcaYrBk+HYGJAvZUqs1/3gskcyDbI0SKJcpX6cFzuejGQJvH4wTYPrh2hTeBcaLE7xOiByoP1PHEBg3Su6tup7ngUS6GRtOfYJoTk5rWN8V2fyAhXEVOgu8QA8A+Idat6MCm0JACkVrAaZgGA/o5m8HhgCZ3x/YidEiYYWOxNsAaswwUbwsVGsvlac1TDqR4J+1w7EXo8Dp+L91N1qRdKDK6IyE4lYxaWIRZDkO4aH8LMVEkTtZV0jNwB9jce4DEZbPA+I9S8A9l8JHN5AByKDh8A/KH+MqRkEsA6cF5GzIo/eP8/4G4RWp56AmKZCZULWA7WD3UIZ6m5Jg+CwaitkHMSBNACfs2RbFiVJsgRPaNuw9OwX26Yzl1oVhrY4SKfKuI2E7ELQWEPT6MGCGfXw0jImKwWf6+A7lHZVpx2V+T1RC7bhC9ALsC/oddqY1oZ9Bhg5bnlkv+GE9hoyKUnEt7FrjZgKwS0gzw5xOA81f2UIDSxdWYea7cFpBhhYSPYhfewMOhGx6Gx4ba3JOqEXbNbh67fRnYOUaeV6sjVoaNXd0EZH7Qoih4chjAi08PM0L4sy8LXZNnmyQK69qKBRq+alQhsgHM1WIAptvxcrgQCyKdzTOSIzFAzEphZ9oex/VR0Aw/e8S8gQd0R8AllPPk7oiLxkC/vYdiSX7ffOmNQkYlZ9w3igbm/qksVZQ02dAfMWqFfa60AcM3UGaqGHigFjkqmPyYRhp6v7Qr5OGnvV1f2G5SIPOz5kqtMkK7IozZJQltt2aLZUCO9r8xEFA/Q1IPd0BgE7Yhi0szQZ1i8gsFkB9Nbqhr2Aa0hOCrdbSQv0V/J9z4qaBjErouahnxQMUJrJzuwAygj268W0F7AKXmDxcDPU1D4R1TGNJe5khRLyIvFONgKXgxVmfqjGRefOBlw/NobBalqxxTBWZ/O4ban/dHR6Q6+z/gA25yho7ZaDrvoLmtD9zFjlSVNxovO9AzYQ2VHRCrI1CvxQZ8po5W+vrsyipbpXqG1H4er8S7DC5qPpiqq3tgNaSW1tyO/2iCKUZAqtybU/XS8Av8C46AGzDhaJSd/TgJByMkLAmOgZUk3eAHKH/tmZTEULdeAVOUWpcWlRu+2AUfu6hQcyxEpzQ/ty1rsGo7+2sHICausBsdr7movxruOwk5UWCjQJLDhju6/Rp9CMNim6RGAAK3QWHA40aT8rotRkWSZEKfNtIy9Klv7yHyrVGJYlys1r6NCrqFjOIbO9qPpKgAE9Bx9QN64+XerrLFhkm7kplDQvZKGAaoosaqDIzLWe1b6RWQRvLnXFZUcpOxUClA19oGOGgMVN9XbdGlTGboCz7mkWK3VzFFf5IpF54RdZ6ociSvIdpLuycH67/lU1jY02pT6sfPtcjaYyixWkAViPxfVSkJVC30L6xgKDIy2GPWTOSweKc7KzMfTCJke3HX0Z3Y21KQf23GGD6MyAemv+OqyBo8jP0BejQaRoX+yEY1ZI618V6Jr+QuJOfMMaGPlHgd2MwNtDaP4AnwJ8oDDW1dR2xzUK5fRpCgPOIIv1ulpzrFQvMJSB/QrV8EoS5alC8xI/a4VODKCB9P23WOdqgdXJev7R4PwIo+wtHEtsOax8q0rDkTaNqAyrsYHhsBs7EMmbecKlf2pbFP9gLtDske6EDqHJ0fM7MxV45Uhkj3EzWj5V4pBBd+DUYcCLERJ1EICulvNw+FEa6QemKHMRRYUJtpczWEUfr+jHIVTfMcZsrXzDxfndoHshN6ohY1N5v6uOFkL1z7wOTvGdRmCCyg2QcvPssDE1hPX+3Fjfb4rBjoaJTC+cHCf5PUodluCk5R345tLRV7Piq5IzIwXUqTNTnq4gzNe++YNkzSITlVJHmQiCON8hayiXvrbvSRMCrGFnDtqkmJvwO5GStaVW5OkE3jEYrhyhKMyMNFyBPqLm68FqXliGARgbJoyoqoN2uTFknd/bpyegJb0AoVM11mvOJShAToin5pZnXiYcGAETDimhrb1qn33MMrV+dmp21p39HoNo6vtpRZaNEM/0sBiLfIG3MW2pk/tEIqOcKVp4jeJtClUsOp4pn2Eqsux4sSnJvbdsw60jcDnNY2f8K845auDEcZH6RVL4URzqZNsXnoVLlfMzrDDazzUoHrsfMRRKcq1r2wF9I5d+il3F3qs6O7sO7czcqiuMKorHiLTMA0LVHrhn8C5A2aFdBhCkLYQUDVFkd+3QzyKA2JCMrEqDFevH+CzhcNztT2tPCcfXu5W+Do7aISEGAgOwC5NAJ9Ls6OtVQAyQ4i+AEUFGPHj/bHRjev4b5/WHruIHMv+fDAUX0Cb5Y3VuZyEA8Q4dah5HIIBMpKQ5LA/Nf7hym5x6c7p2tgcXSAQNL7mBjfo2iaY3Z4MggGJVHTtcRMw5CMj/PUZy663yo5Bl7ydBAxo+qWNrbVFyE3e+sgGTo27BQMQwbF9mZRGE2t+O+cfhKvhdV72ilByMtgG6Q2x9YTXKNnkI3EgjB+4dVFODouDiwIjAnxs15FRjtIe4GBq83COuxLw5UjkoPlYBrKM2kgxBeRayKLIy9ZMk2xH3SxvpT8DygLsB/z+pTqO1BGuDspvVfoNDA/VVAtBC9AdDrMC6bWmXjMYvevz79myN2vxa1aBYA5ogiGn8k1QmEOMfkKlmLQKPmHNvLe5OoZfYp+/E1Wg7o4+BMu/A/gTtDnaxHbIHergqzGQK23rYIrk4O1NyAGIcfU8uXG2cU8M5lWzXwipzbof9406sknZ7ZyNQGHWaLSyXRxHmhobwqBhYBNK6cuISEdjtKxs6YQ+9hZDPxmD482zlJ9aak3/lmj2Mr1QU+6WSSRDkWW5KfycpxF96rN6fXzmYDbLBvJoe/oYZX0AXD4iFyQXJkd22brt+MipYHqFH5HoBAjj/ObmMqFZffZy1OKuO6razpgaQ3Ki6J0lilavrGzAHjI3c+NDDHJTbtkipUt7fOIjZet11fW+qrCh9dAsGJg5VocIsyEoR78ifIL7zDD7bLDHOAKUs09wgcH+P/7Ww6y/tADgKhEhnw0rpzHcPSqEsSZupWZXiVJ2uCv6xNeR2DUrRm9XCCPs869WBUculs8oTthvBLLLLc+VCCjSLkfL3vTPtiQToHxpdaeFs7NgIDTJwQ8M3Kx1x1EmP+W6pjgOZGZOJItmJU6wTg9WZzD+QZb43ABw5cVYYKuIKvTVdhagiHN1lFK9Tl95LLOLHOtZMMLa4xfygOApoBgkMtamnr235Mc/D9VNWH1c5jIdlQOpnYWK0yOIIbIHteYdLj8wHEIu0zyiFszKcd3vDdJMQZdLg1a1FepjR1aHEOyuUvTWpBmtDseBjjE6V0WNSXAcOYVK7CAcxYoMpty3mBE8sw59RbeIObfPhzpdyGNYmgRE6BlBmTGy02lv69D5AYwV/BNZ9gVIFrUg2KzBlHHUAF5AL+46SgNCjhBOB/2JTs6nN2sLpuZau64zHoyksgc4AlIeJAjM6TPNt6Blkyb30sWm4LqydKzSCCzKnyQJivwG6fsZkx4EdEm337GyhVR4wQZoRq2Munu/PkCCB0Wnzu5y0nrN7e3ZFrHMqjmLwPPQjmfoyL/MkDba9xWm0REI/gXizqosTW19h+k+EIGAxQd/YBGSWl1RaTimws3Ksvh69XzAF7wzICnd2xYpsbmtOeTtT41waBQPIVEyaRGiFbhfUs/AFELtLpUyTxzFrFsby0zr9+6irtszLRKcAEUwoUn8n9TsTqwQKYP6uhlJPAOGrhsKnlJfDEyJcc0F5TqFBsrPUWeVtjioN0w2KEzxjMALhJLVgagy12AYwEdqR9GB9xOJSeGdDbsoLYUt0vDOVsEFmN7Ua+0qeHuUxlZSxiLTSmRAK9tweplrqke86Btu5QuOoYeexVucLMdUZ1P/rA8Lr0yhyFMj+0/Vc6aqskCIYGMZkFeAt4DMMOCFCdj79qrN5B9q8oPCZzENajv5bEK+3ByCSrq7nB5tb/wTKzPNoaB66dIy2OgebgMEa0NSU3jbmOJeUoTACb2gTIZYNoXEqjfBgaTgywUHF5TCdIOG8Rx7PXRvDzQlQss6pmVEA9euuH0bHCk7pSxpbDoPTLJgsnTUDFmvB2V8kyGzoZjEairch/2pdoXtU1avSy+7G4a8rRYtK87ktM3fY9TRhNJ7iuEiLoTNRZrm0uCSj9GfefMAXtIhrrHU4aqcyXwQgXZK0CON4D3PEG1CLThuN1riiVUJ6W2IoG6yzqodLO0scMRbKyy2vAxe29ZfTig/OKpF+UmaFjqPICLNrZy+91cBA+qxAx2PCYq7QeRB56oIR34sLzFgMbc4kIuor4qEWcBNZ1YiXOMReGQZOqcNGlFdnBlv/xbZI+82mDQDWxu8qr83YE3qWWV/3pnvhuC8YxS+cM8KtmWZMG0HhzFxaDmbRAtmIMKPGRqyHMQHbjg9auNgMZ0rRQbdRuPw4RVPmuQK2CNXgHmxAjnteceXR9ZNFEPhCiCDO06hI0x3v5nIBf1XD8Aqr8mt9PefkqaKsUcy6okNIL1VfobDmRHJnxSjMTIGt9cBhSTxL4dxEDx6sAyYbMVna8oFWpgYl54rj156jD7+3Ocbknz2J/kzmZ6Q2YfO3MSE3W3xxjIyGbce6Ipd/VNPlpQ6iOFaRymAf+9u5dsmSXB/aM4AaDLOAougUEMceczrRURoKjOFijqfauvbWjAHKACO+NYlrNKNOxoAqn8co7WdCntwVe3XDYGqJ4vQBFyQ2m9q5i3iRaOCGcLBrUklx9DhIpCIZCB1IVcSBSHeSm/0Va3XPmMR3Mje0IKd98XQ9s+SjJK4L5n3at6jWRTQripWRjvgROQwLL3aSKp7HNl18zxnctbKJgWDb3kf9j1pXYVgGUSTLoghCFWTxzhnCpeP7V0Rr1gM9N5b6FscLTz/jrAl1UHBtTF0mE6LqT/z4O02XytokOZzZosWpMLddzfx6xEe/L2vDnuIqD5gzM0ueZzU75V3YoWIclLugQKK1ZzjXbZ1SedRmN5HOyyLKQ4DmZZ5ukzRbxRJ+xmgshyN7T34TjBQ4G9XzmdH+QrlX+GF8V1wvvbPkXETA5rRe7VmLp6oEcf+r0V0L5vpdc25751U3nAjafZpwVzzKUsGuXYx4UgRfFn8RkcnCIohlkpWRKbftmWh1ggWm/QuleHCu9/cgKbqq7V06sMUe4xFNDJt8DXuoxqRIijCxY+sbBiVMRGpJt811+Gw7T5jkIh6TVTucXTKeoQQIV+HBOjpnYrHajbNY1UJdRzYkM/fN4XKO/XrtZSNeLw6bPrEOlB8apWQCWnQvGSvYyPS1UfdvyRXy4P1cWSPkQ9sszhBcakTllNEIFh8hjsDr2HtIDhlibszhBijUU0NQgJ9mRjb3MnYgnb09+mjYI/OT9V+CWh1O3M2DXTMcCKdg85nQj0OnuJ/x6ANtdcrflL4dwsLlSWPSrc2eQCpwo/aomYv6zIDVrBeGVzkuI4iXoaty2DarNLr0qK2fa51maWrSAuS0v7M3kiBdh8cKF7Hl8AjgEEvf3waEhY/e97Xq1Cg2llKai8OTKwzbYyqP4R4sMPMMzTohvewKv9g1owGtsqnuCdC/gFU9EYAevw1kkATGT9Pc90sV7h3yWjqGvmvznCMhUs5VLrAVJ/eggMPw1/1HmzJLJSKa2Jg4YHNr+gnGu2qcK7UGweNXHAiLlylZnhLUVkdYj5IkMCZOpFBlKopAB9sh0yW6+4lS8ijaOwsFg8yFZbMhZhklnNP76IW+TQfmHN/fga2th8EFjH9cLD0Xo6Wm5kYnuO1xI0k4Sw/O1sRZkCRGpMoYmaudxLFoFQhjIquuwNjhD+ShVHhAGU+7D3gsfOBEEULjxAn27ZioAvO4X3k5FluuuHuLAJ56syaFa2nlsTo4ddjvkZKmzIsyTFUhj8B4Ctc64KlfdG99uUnszps92qMoHbk41XS5CaZoRuOpNHfeiNqa5e6TN3N+HtlOcyy8mqs4OFmtQ11kReSnWW7EjitcroI8ocCgU60xaZhszq7S9jYKtqL7MdkpxyMTQ3vh1R5rPbUDnXgfi3+wp5FHuzedCtOiz9teboPlUKZyoAxK1l+zYbg4td2KTXv7fHhsk3BZGvggb2RcqCD2/W23phRidQoWAwaX67ApBsfTfuS1ABmhnmZIU2MKybrGrvCzmXnEHtDOZy2VbebwkyRUMouiMlTpzlGBJNqY4o0AqtKc9k75CcucT1RbHOjCv1hpoL17Iac0H/7FCHjoA9ZDfz4QYmDLvqQe4jhkDxeQqgQc3J94oandkSxjiygiKJNMaefMXOUtBfIgVcrM13EZFcIHs7AU2Z5De0M3onoTyafVn1jqP5jaUBFXII2C/wELLG912YnYHwUEIjFKxn6WaAAFRbBzjjC+u73gD3iQkQ6Fmtr6XoArekxRsOG0kM8/0HzleLoDk0Q6zImk825U26lwdeboyWzmUx3iHGzQw7TeL1MAoa/zLAzKNBd+GIq9iMUStr/XyrobfHZX9mxnVxaCjy+Xwovf8lrZJhx849pftnODsChTbUopIh2LfCd4Hy+37o/26MvNNGNKDl22ABsOPXh8BddJdS/GXvbkAhyce+QO2tgSqNZ65065Nl11mZpC5DP7uvhmt+f8gMxmOTL4JsOCrYLERQ5KN4zVNo+PckGUmERqzFCJ0C+/jfekn6588tbkEsKeR7kgR5qVGfATpi+4kokrMzeGbC3iC661BeeOsgPYM1kiMt8kAGzyHewqVgfx3uOlKvqKthSeCYQBAR6lTUYu28DrTxVebkP58RWY74ZkNOWNxe4jXxqzSO7Dkv14GMS1a2+0Wk3wKGSL8yCTidIhovSdtRKrOPav6loTEMELEz6yhwoxSqdet7w0j94P7iKxEWFjVNtzB0AiOdZ1Jxxs6jwnQFDYC7ZFFESuoBPydFYU5XuYTY0ssa57O1q51Baea6iQOdRtrdCiNDkqLPKykEks0jLOQhnIHe5YCfeqLHFclKJiz9uid8xlv3cGr1+jOMWfG/Qeo1Ont0gmIGxjVZW9+60BY95zhVisJO505FSWM5jD8V3eDsNSrC5Ku2wYlhD1jp/RP0ioJBJhLlQAqlAlOtoBy2LluVVFy4o9G08XsZ8W6BCNr3C158DFvra4ZXSAIMQx0GFzt1eio6ocIFCR+EKlqUj8PCx3sl+W12iY3BSFmlJX+ktNYZzZUWt3+Rqh14DuOEBPqavKuv6sury2FACkgxXmCYfj5W2kx6mJDbl3VITnMizixAeoGIsoV/GOWFgitZhc3dZTLW3OKpkWfcWBcnf8EP2Lj+MRXpvbeiYLDW13ChSu9b1tm412qqJ2Wl7x6GGtFZQG/udLUURmB7KvHFe/LUyzowadXFlfZNI1+xbd3GicduK2XfeZwWxZdis1f9Srk4MFJ/EodJlmUSZ2Mn6DFdYDW73BAJrNdZuOb+OVMPQRmYdUvCs1nomJpzPd9jYQgsIz+TU78W2bsGnOrvCXsUYp4ygRZZiKMs/Lvan64epocEPSiHZ5b8UtJ2zSse4OACwlrI9YMI7Ht5yLmi1rDeM0Hr1/xEu4LAyK7xqP06kl68afxM2MW8ZCSDFNA2Z5P3Cm7ZdhijyTqfRNWEoTpCbbTlGWYfL3hNCY/BBP8Ne2coeM7d0AE/J26TqT5901x4nLIzOKcCOQedgmBCQThFFQpGHsF/G2ISxXQdxf8AwpJc8884W+GGd2t78WiO/55D0OFEA0xdQQml2fAGohV2Tju7Jur527+otvJLhQsizmwywKjTegzVu78yMtuhpT3V3wV2n9ZfuuCNJChCILojAIRbGTFrCkUuR1dI8BWwE0egHqEHHf+HI6e5mr/nmRuM/u4UVx0jiLRtf1P5vks+Mt1VmqytwIMJTSvQtfs/igVJntiMDfExwToMYw0Y3ArxcHlOHJR05mgiWUi2Y+JUhsQGUqPAkUdGLZjqwLEUOaI1KdDW7tVjoKwovQJL6K/CBTgdiJFa6kDcXxYdeYClPUAr4OY4zY20tzHwB1LT+QpdzgvSV8c8Xyq81fmqMy/s76aOpx1Q8nNXHl1UY5ulPSQIRFKYvcBCoo4+2oU7zyRQd/d1+0FEd80XvtOd7YBiryqDpKkkBEmZYmDQVY8du2R3jvgPuL+mvb0VVowzuyyUGoAQnISn/rRWKMPCwuzw7G0noyet93z2BTKXs00nxEC94gTCfQC3pmeYGUu/GIz4dxc59P2N+cvFJFFvppnIgoSaIs3NE0S+gSOoCNUMIB708AeblA6J9C8s5AOITcj/poAGskSZrrPFRRLnascJlG/0t4tpaqqFrCGWLbgWfJXa00PobO7oQqyc8NzFZWnxKk4qhCSpOwKExagBzN0tLfQ/QrSWq6zoau6b4Xvk/q2pnx8lSOs7cU0D3hkUg7Z/y1gUfvT5RCjLJVa7q1ip27y5Ysgqfq9wQacwVWtVxUi67gwKpfBtZMUposzaXIs1KIdBvOhkvu4nwSowZGaidDN8zR9Y18w9EK5X53LRDUndQZTyCqgnEdXnbwF8Qc9fJy1kAknA2KJwvddttshsuTD24MorJkzqKpCf7NA/NC17Qh+l1SeCxHf9jA+n1XtJG5t8VlQ/OWv1CE55HOjFCRiEM/EfF20qOUdwYYnzqClbb3Z07XK4Ccng5A2Bt+0YK097PiQRqUwqPXcnaCFqpTSufddTRVv3Ww86j7IctFkfnSFzlYB+n25YYyWp0ZvOIdqpTrgwnSTxz5Awz/XGnacuw9ZKR/Vywciw18dN9eQX1B5pz/dsXsSlHKmsIm2obvUUcjaHF9tJ58h1SY+tpzIMqjsilUKtV4WURZ6ESJbVMg9Dc2IQpHSiFE6LIHi53t23vBZ0HvFFaW0afL4qG2Gs9CTC6KFUTmO50+IbyP6vlIpCpVflzKtIhKoXcib+F2zCXZD7lYy5GvEpt//rnC6MQYYSE/xGfCMba/tVPyqMcpyXWkghR0fSr9Ii133DAHQjGE4TIXaXmlKyTc7yuEoXtvf3Lip2EjdKPGwM1dfMbdNziPz0x9fPbez53IeSAFKJ8kLZMg2wmsLhWzjP/n8D5IP+OavMfsapk5cpcWEh9OnAuQfY0ulUxFGu/9/om/xumLpE66kcrupOHV2llVN7vV8sMY9iSBQHcH4PkTPLjSP4zXTjvBjzaSqhZJSDjjmXTMbIn1BZrzWwrnfWBK0hh7/TJukCaI8GbQspB+ptS2RRMFq+wSewto6f0RYAkLMMygo7NQfG82v4wDT/f8/s4xQoX5bhXbiIu5z256/DJ3SJRqmaV+UBq/9FO1EzXOsp2ocbASYXyYf54psIgjz+RWtqiT3qcbzDMNdkTYYUM9BwyTmziNwzzGs+I7XC7+P6LJxI/+t2hSZOITaJKkapQsS+CHMRihhr8T5JRhtOGmjY5q1bQ0sVYY4yjyQuttyzIWK9j5goYZ/D8SdPnmuytjg6fXs/3tBqtYppJhNt3V+W68rPNfFR5jmqdzjN+xkv1M5LVNtbtAKzoqW6IyEWWUpaoAyB3s/NyMTNO1GLaCEa8ttUcIBmuaWbn4NQbi/4pn1zEly6YBfG3vbCaGosPoHZ6NH/gQMkoodziQ25oBKXwk8UPVrBcb//35IwfbnpLQFBIPwkipQ7ODMKV/H6G2adt8hdg8Tp1gnk49jCHpMZNbRBulo/Cu9KhAFoX50B11vJmYlhz2rmdSgTXl5wAsgmwnMU2umdtmVCYoFntnXtPPZLhjFHiw2AFul4CZrsP0VNtC4C8OEARaZEmUB0km/UgmO3ahn6wWrXriC0Jmd4vSjdN0pwjGPW4uBDf/kYlX/o5OighFrSbWzmx5vobmJxs2okTV5e8ijDX4alOqRDfizHCC0bOQFI9mWSE3X5iZF8RJrAsgkxBBlukd7BytRJk9fHU9469M0tkeLxWZH6Vs55Hj80/qxueFKCn7FSAj3ZPpLhyYfnMojlNYIq7ZD+qVfoFpjTD464UvxKZLn+edu4v7sfECRqk6hn0cU7idWmoY3mHTd1dZJXQq7t/hraXWf6CaoHvtocDf3nwPmLY2P3zz2/97/8uvFEsyfl4KE5mk8E1WhDlsGUCgWsqwDAXgrxSs9cQwMWHMqvqGD8VA3ST0AyGBs1OA7EA0PNcaR2nqxwWY+gVg3CjP0jh+A8v3BqShOSus9k/i3T/8y4d3hAt6Y+1I8smmaGBU5+v53W8/vpdRTPcoeP+Jx5TR1Kzo/h48ee/yAJkh/uu/AYJAYjQ=')
COHORT=unpack('eNq1W1mP5EZy/ivEYAb70ltLJm+9GKOVZUlrWYudAdawYRjJzGRXqlkkzaNrag3/940jySLZLIsewBAgTCXzivuLiOx//+93Wg6yN8O7b7x3z/0le3n35L2zev75TeSrMI2MzP008ctS4Pems8+2ltV/2lqbLzA3zWG07ZpLSxv91JxrT+qmHXpPerp5PnneD8Yb5IvpveFscMgbGvrnc9c0F9M9edezVWdPNT0seh/4Pqz5fP/uNWVput472wtsGfofPG171Yz14JVN5xXG1s/woTZXT439gEvw0ObqXUbYVjfuZNoO5+JB/4DEDLJ7JvpT/+T/z5P32yzxSx0XRslQ6jDRZbDLksDPkhVTfpadMpXXjTUypbDqpirjwVU7c/J+sL13kbb2YL4eFfKtA4adOwP/v7Vw+aaENcC/b7yfP3/75H37878+ebLW3ufOvLwAQSfiVttZZXBuUxuc6MG+72PfpwX44yyrkhgxz5Q174ibyXk7OoxWRzGI4seadrw09XB+8hwpfVPhkqEZZIU7hb7Pd/QkTHymY/jyle0Ho4FMPL2hC1y8qwESV+f1fIsg/sAf8c40zsR1ph/caj6bD6OpSCuedVrI3OrppkZ2tVeCKLzeVBUfR/zvPT12lu4qByZvrRRBGIj4qGKUhS9FJBOjU5HnWbSrGFkUrPTin2TRWbjjVdYgdrCKYrwBV5XsSLPRFICvLJ32jFKg4QwtZFp7lixmn4m8Nh3xtKmBOebF1LqnDc4gz6JrgH+d9yxfDVnTewHL4NizqVocWHAQ+GFubDvI8sFWFViY0WsWxYl/lEFZHJgkCvMyikwcG7NvOdmKP3+spAUBX8h5gOl75hkou5jKDJ55Nd0NrtnVSC45gs7Il1L2w2T9skYK/mZqMiFYCwqDdPRAkQGZg81FxKV+4w2OOoMoiYUsRRAWKZhKsktSnKx9waezrZozmlcUeTfQTrgc6PPQaHmDi4O1pTz8hIyn+xazO7AXZASvkD0JtTbt2VwdyThul+O870avj4pMhKUu/UyZDPy/MHqfPn9F3j9XzU2TUoJzQXGVsruwk1Pg5V9QGOzojCdiwVJpQSnhmqyohgy1Z0dB+i54AkrytPHrfBwqCGr8vBkKdU20yKODVGttoiTKUpGKUvvRAxcvxFqsH71C9lah0lk1dmgyskNfPnjvwxgjHHh8sGKj33yOwJo/esp0A0aBXlZAiQt5AoId/NOzpXdrRg9I8Qqw4ZP3aWzbpjfenyBW1MDZBn6QB9ENzSBWXpsR9aTqG/DCwCH4LLX2amkriDEDcUwCJ8E40C2UYEgG9W4VjkNiKK5Z8J53RjtqQWxwa1lVa37nR5VMAbPjuEwSbcJQKH+X3WG21jIgG9SraLquuRoNl3xKYp+UpbSvLlYRc2GWrQdD4aOTA0W9wP8ADIS7o5YCT1rSPM/81wixDMIXAovJu9TI7LXK9bTy1UxLkT070SPzw4McCEITRL4sskRLKfN438yiFQMwJn7XjMVQooPsMcZ2DBx0Z1/RI57BT4L7AFVIvRc7DGh4hMuAXxQpMJRWjUKaa3uR5BarAY3nDCoFXy3SRQEDJlXEKlsDjy9smhQ2edpwhRjSjKCPSg49c+dXgGLwLz0dzizv5a3nlX+WgzrfnugaQARMhrVPJBNcBQ4PQQp+rcdLQShwvr/b8smZ1efOfrGGt5ouvNgqEKd7PJhYMaPCOxdLebHVzathKlKwFmd02GmmfhLlKkwMeEyj9t1HkuUbgFg3FVru+HweWIerBtBK27QcvstmRKpqbY3jH5onKP5JIEQr2ZCnNZPt+qcI7OY8Wy0Gk8DfbA0D08a0J+CArwuHkR+qINKpCUWSyXAfAgVBvtbk71EXUWa9sqYGkatK9mBlw6hBO77tpL55FYI4o5lwovQKYNag4yFAaCQJO/IB2lTgPhkyV02jWevYBS43kaDtsnthF7COrcQqWoy/JB918v4oKzVW6ELWOrk90dkFmBCdAJRc7YXSjtVtX81pHZTFYSTlh3kRZ5EMClWkssz307JwizpMJVcKBuC4YlTZO7QJLpSYi78Abf2evBpcG2JVVV1MPWCcEh/uHnVCJ+qMZGhaSDuMtR2W4QLnuSugy6Q5vDsuCZ3D3kTsOD6qeGmqlCxj/J/wxQOT2+Cwv8hbzWqT+AgjMTOtKoRjIMAaAmvfW9RANBH3g2aHf4g2usJQGX59hMBeWUn2+JH+yegcZLu0QoqdxAJylJayGkCqRlmN+slRemiADWgXkyemQ2pNamU4WoG3mtOtjdHCmYfxTqJKEWVRrkUaJPvAPBfRBu6Q1fx+aODGfYWI49K8AlMobsLAtXK8a+oKBQ7uiOLKmTLLsUVrQJg6wC4AXr8nlbsYzibBtCC3Af9uh0XlgI+JMNJ0FI0I9lGm2dkWxHet3S8DwaAFsInnzHvhZVRlLwUgNf4EetwsL/Gx70eyVMQ1sNwdKy9Le3f7zeAJz8ZAhPkELwBHA7bxdsUZw2M9Xwxxw+Lic0lkhHvaksD/lBYq0EyjBk9E82bAvow5sZX7QbHHIgvzLEzjQmZ+UqgHlYx0Dby+M3WN8kO0GXgtXRZJHAdEDmT/pAEE1j3VNbbvmQpk0NnlclwTwnTyLsfkzZp7FUiRUmG5xAPwDIh3qNgYJeYSAFIsSINSQMifMU3eXgyBM46fuQhRYqLFxQQ3wQVMyBF83BSXbwOnHeb4SNBv7MDt9Xhxmt7fj9tIJDsoEZmbOEhkUgZJEKbFg8Qj8PMNEsTo5UojVwB9tce4DG6rXgbE+i2A/RuBwyvEQFTwCPQH/Y8xFYMAjoHLKWIx5eT9daHf4LQ6+QzMNBaDCWUPtA8eC3PouDUPwsOhTYkkTEJhAD7n6b4vStN0DZ4wt2Hv2a/MpjNtJZUhEwfvZM1kSKguBI01bI0VLKCoh0GnmBwUfF6DY+jtbKcnLvM4cQvM8BX4BdgX4joZpsthXwBGziaP6jecMV9DJSWP+D6ZdiOlQnALyLNDHM5XLW4MoUGlrSuouROmyAAXiyg/pI+dwSIiTl1cr6k0ZSc0wGkdDr+P3xRImVfTSW4FXc2+udpcqN1A5OgwhAlCHfhFVpSqDH1t9lOePBTbKipEVFu/WswBojltBaaQ+b06DwSQTaJNF4jM0DGQmjr0hb7/JjsAhh/ZSigRn5j4DL6eapxwEFXJVvmxO0is9++nZFKTi9mcDfeBtb2pSnZnNW11AcyrMK4040Aacz8MwkIPC0PGJPcz7ikMF12nL1TrpLvbrup3MhdxuPAhMp2lucrjLE8jUe7nofk6IHyszBd0DHDWgNrTGQTsiGEwz9KUWL+Cw+YA0LusG2wBZUhFislaKQr0I9W+F1NNjZgVUfPQ3wMMcJrZzuoAwQjstTVNC1nBKwgPjaGi/Ympk9I45t6zUEJe5N4pR+B5IGHWBzsLnQ8bUH6cDEPWtFGLYV7O6XHT0PnZXPSGUxfnAWwu0NE6k4Oj+hZT6H6RrDLRNJ34/LYAGwb5UdcKvjUO/Ujn0mjp70tX5PE63EuMtrNznepLIGHzxXTK9i53wCypqQzV3U7oQsmnkEzG/jy2AL8guegBsw4OiQnf04CQCkpCIJnoGVLdqwFUDv1lSplUA2tgiIqitLlwqN0dwKh9u8MTJWKluWJ+uThdQ9JfOVh5B2rbC3HY+x1PY6vjtpPzFhIiCQicsd3vsKZQzzkplkTgAht0Fh5uNGk/V3Fm8jwPglIU+0lenK7r5d9ZWRv2JXKia+iwqijZz6GyvcpqJMCAlYPPGBs3n9pqXDSL3DZXiZ7mlTIUCE2xQw3UmRmrxeorpUUw0laW585e9j4JUDacgYUZAhZX2Tu51RiMpwsujicqNuHmKK7yg1QUyld55kdBnBYPkO4mw/k0/k3Wtes2ZT5Ivnmxc6rMbgV5ANmjGltFWQp9i+gbOwzutBiukE1VOgic9zwbWy+ccnT73Ze53FiZcuDKHW6IxQxYt9WvwxE4jv0cazEaXIr2gwftmA3S+jcJsaZvyd0Ff+AIjPojIW9G4O0hNH+CTyH+oDbWaCpncbVEP32+twEXkMVVXV06VspXuMrAdQU73MijPFtML/GzlljEAB4I33+Pa0YHrM6u8o8J5xe4Ze/gWOrm4eKrLQ132jSiMlzGCcaE3biASNXMM4r+uWnQ/UO6QNQj3wkdwpZz5XeRKrDkyGXPfTMSnyzxyhA7kHS48OqGxB0EoBtxHm4/CiP80KiyCOJYmXBfnOGm+zhiHYdQfccYs3H+DYXzq8HyQmFkTcmm9H6VHQlC9i8shynwnWdggsENkHL9MmFj2gjX/VK72m+GzY6amUwDkx8n/z17HfbgFOUn8M2z4w+L6ZuZiyQFwumUpjyP4My3tfm3bO1f5aW9s5V+fhOKMA2Nn2WF75cyelTSXAfNb5uiuBEeFAKyfaO4hY9FVlZlLCL2WJ5/89EBRJoRE6GkeQtNAg5ul7FnmEHD9itehBue99SQ3PGmYXOUJaExSSoCWWaBCnW4X2xbI/cfKQBhl4q0v+NMM0wEYIkbW4CIU0awJ0hqHPhlRPtrY7lsa9wWC4C0mEYKQdvNpX134g4kzrOD1JokD9PUBJk0RhTygZuM1+S6zEJ2SmrjfUcxQGI7Dnu7AzZBsbgBigA+RrMmuNHpDcAAdLyVvJinrSU+jWImQKehQ1zstOnZHyTdL6NYClMWqowyqR68/llT/o+UwWB5DeHUq+5drpgmU3X15AovHdUB5f0pDwKSeK7BTtU12muRqVIbdNl9c2TOkze0BgeJ1TrSChCXn+WFCYp9jCA2b52iAHvalUaITC9ROqvd2wsuLfVzlllggWBoWpb2vOq5Gai/O0//7HpvVIiB/wXZfTIJfbn32gzWV7nPAx9essdcXGNyn84U6+a6sZDwIOPyLPTB34hEyRCiotxnXBBsej4SYkQLue+eG5xr21SfBR8hn2djoH5e8mbFQ+fn4hCpB+yzCQNHlcNP00hCbhSXkcweJMZpvEMiAlvMiDnJ6yiTXSGcAlAlv3DCf3HQQNjTElLkVhcWqiI/BvV+xpzkR8RJMLekE5IkojwDDQHgs+3PLGjad2bLvCO6CEJeUk+vLjaQIxQHuVLmvk7KWAU+QMgyyB+8k/B3YiOGtyD938NfsI5/QNpgSSuQR+H/QQXWb5hmKL1JJg4SHaRGisTPUw2gQIUPqubJm17991i2pxYI4GF86yc70Ioe30+4LmTE2T7RK+ZaBlYPwauw9fLqKYTLCzctF5Tf15Dm4IYegtivCwCRr4s8CsusCPwoCuQDAa8bjR+1vBA9kOTIFq4DFP0EftkaF8KmwbXz4lGWldvi1YE8Xv11lhtGqsy0KUUQ6yQo0gfiWpvuD67QczWckvSG2d+AwYEFugenkJpAQsJPGzH9oIbDOCzKSm4GhrVeYjcJVHKsO9vet0Lks/i6+ubMc1kO2p1HuYY7cM4z0pC9AmZofI2NmSdHtSBOTSp0lId5rLM028d7wl8XIn7CWh5pQeCqLy1qpHEw7dOANUFS9fvMdJpzWmo0ryK94FV7cO6oOhRa52mQ+yYFYFM8wK7Bpuz8EZ8Q6bEfOosVcLgQ4FEyMqpYhpChW3zKRdmgrcfBkI++IPZLpo/8RGqp9TSzn0sf077u/eaGwKOQLSnCXKRSR4jSH8gKMsQVfX+WY0VABJ8HfEGNYYzSyRuWlymxxsejQMeZ9PG76dnsjLDxvaI3lTtiMa+d8nmXKPJrHXp2BGYRh/E0cXLy1BlB/x7l903WWHcaxVmFQVRE7ZOO6gPA5Os2oMVZetRZFKUSaRJkZZJHIhQPtGPj3G1Z4r1QjlN3CZ+YS8e6zuBjY3qk/UuN1QF859g7JBMStnGhyr10rs0XKl3SJHYr6dQLuM8lzxFF81jRDMPara5mY9ifC0mymjpIv/0udJdRaRxERSBDCIUy1fEDsBxEGzVTDQf2fK6lUa8M+RDPQyjtJXBxww638IMJB20NHFi/sZX4aCgHCKRSP5BZFqR+Ee2junCN+P9iCqOUvDct+rayaDmLxuL01JjQa0gd/RYEMy3lWH+RHRWg+c07LVhQfX+qTHGcttjxe0ddeCEilaQ+QMUkiAuZPHALa6SWUMP9xWXnaP7avbfvLbd6pmI7vto6zQ0rmke9LSAfc3dqfmzjvdubk3ZaIh/svNHRw1ErLA3854tAxeYBZA/XQevTKjU7mtCJTfZFKV39OKNbJo13S9zP637jMnuZ3SbMH63qFJDBCWz8lVke54H/wPdtsB7k6jX+VQE9KosXzUp8AEUfUXkoxE+z5r8YSe4dTPf2haDwwn8t+ptuC1eknCZ/nWqUIonToIyyoCyK8hGpfrRphNXkjcjKe+duQc3dG8prBwDWdP0CCybJPEpTwny9apjJOHn/gk9OHQxK3myeZPed3Kupu7tZaMs8CTmm6cLs7wcukn4dpihykQnfRKUwYWby/T8CEFH6/wmhDaDk5A5/3S5vkLHrhN+R9/R3BdMTu/t23LSclTGIdpp1h3NCQDJhFIcqixJfJfuJsAj97WOGG1bACvnCf76GHaTpb50U4nvuM+NFAUQrbHEiNBufAWqhVuTzWFk1Yzc9dOX+O5ZWQFUCsZ40v/dd7vamjrQ6au5BYe/ZvU38OrtTYaaCKMjDOAqjQO2nzpvnZjEwQ89ZAN0+gHCIuG8evHcaCtm/mOFNeXg1nSLOatPt+jV1CVD3H38Hawu8ig==')
RAW=unpack('eNrtXY1z3Lax/1cwqTOtFenMz7uT5vm9ceKmdZrkZWpP05mqo4IkqGN1R94jeZJVj//3t7sA+E3eUbpTEvs601hHAgtg90dgsVjs/uPDF6skEMsvLpj84+rVF6fsi2STrzc5PnyXsEDkIl1FsWD5QjA/yXKWhPT3dZok8OKa8RCK0KMgyvxkE+en7E6wWIiA5QkLk+UyucP3mWBZLtbZxWV8GZsT9g1f+pslz0WtMuMr/GdyGVsT9nbj5Sn3864SLEyTFb1I0ug6ivmS+jdB6t9GaQbdWIr89xnzB9th75oUWiOMMvbMNIxTxuOgTgPe2MaXE/Yz8IbHLIyaBVRXvXu22izzaL28R4KtTmOBWr21SH0R5/xasD+I9+tUZBmwk2eMg0j8aMWXz4mNl5f/wP/k4n3+4XW90Y/sJYpzveCZQGlmyQZo0qM08ZZidRUF+Pw6W81vLhzDt2eO4OfGbGqEoYXlsjxJRXC15J4ECV950fUm2WRffDxlR/B84uBh0GsGb6OVyJgxsQ14ZBtY4Z9Y+UeoRbLKxrOZRKuGW8NoiEWi/xxR+oujlN7AoJMwFCkiB6BaEAaKSS9eiTUasPtFKxsJVrYfrNYgmoprEYsUeP3LgPQHDuxagszzJIcuCp7GIMBM9j4TyyWK04v8e38JrAg2qZ4zVkmcL2q4raMm3qw8EjjQ9Bcsv18L/OFFN8C3ZBloaFRmKtmFVNyKeCNkD4q6VZBCm0Wdbc1c0IdzcvIuFTc3ciw3Ijs5uVAjl31Z8GWoQSx7QcWgUcYYzWv0L6Hlx6LFOk2EzeVlCHj4YBvGxw8W4cg1VN1/jlg9jTBwPeFzmwf2NAjNo8h/myKnGfnk5Osf/t7dB9P9cnwXCmpNjWzbavebQdWuC+A2IMgV8XVtQe7qXas2VLRxKZXzd70wXy6pbKYLZ4TTk5O30FFmXozt48kJVj+DJRA7VhYn0hLPsPQYWKQPhqrU5eUfFBhNgqJev6C2guXl8zqZAkolBdO9vPxSVx25Uh0WYAuAxGoDPATAiHv2J+6lEX5IeQQSQehkDwKRBERVi/F5Wugt6wU01o8jtdRDxXqn/E2aghICCgtwT6OpoisM14Y6pfZQ6Z9Wc4FBVW6kwBs57mybKrfDYC+UIg8aEODmKzaX6DHmI9av0DO45fCpCGbW+fncOQLhUwKC0dy1yQaCB7OlNt5nLjZHo75LUprkkhjkKW5EHGTUxUWUjVr4jnD8LOD4NjltEpP70FMoM9kbXEcuik8APh7fsyD5j4hJJxHXoKp8s+QR8P8OMSg4MCJmDn1E2WM1rQp1IJwhZeAVC/i9hOPYitgpicQxNdVgoKIDFZP4VqRVBDcqRzGMVXJoV2VtYJhSY1MvVxzVJ87sEcvj3DXF1LHPQ8cRrivEERKfJiSwCktWAiYcBjvc9B4nRdxKkAktFSGI/JRlC0XPlk2soTnJuqJX1sWDeDhqkTyC8gjKJwfl9uXzsLAkTUMu7gwtxtD3t4tomSxAQYjFeiHupJ035ylZmgMBn5E2XNzyNOIeWjNQUclEnuObzVoaDf5vw/MINFePo/0Z/sCnUQzsXckX19GtiImh34MoYOvN3sN/nzNvl25RvTcxm7F7wVP4WIr3WJ4+J0+pTMxxQF2agbLkmtqCXa0J2rQi2ln7PVVW9V75fpIGxIBEqmZSZKcwsjH2TWfqWjy0TNubOa4xPQr11yrUZl80HSiXCvgvGZhgW0DW3azSbG02uZNHkndppKYKzUjVHRjDy4YGtdWYeYTQEULDENrBYHlQEBVb4e+XyX2gVmAQcLGg4nL4CNtAY72FAQYbHwhp0v3mgdp5i7bNE1OB0KDmsWPLUvWgceNefxH5N6j1qYLMcq26WiElzTj8f4YPSGPgikFZMqJp2PIrRGAjl5cjViXLDoPQmPtiPpsalgiOaPjk0EBHaVT2BdAtj0hmxTts7gVWpaO82dSp15Jv1Kx3cjJqyfr88FXWaWz7umrb3e0OorNa/7OF6chF77AwbBhg/wLcgVU72cDivub3gLmUeUm+AD6teByBMiNIS+LAi0D+vIvgdcyjJXD2YY5RVTty0QqRrfrpdJkBqjXrHWrUtIdq6s7LXf+roO7XBGPBwlmrh8iHolGlR/WQHTSmqzojVr4gEM7Umc+smRUGhmMeIfFpQmLL8IF1va59RS08UrHdCfs5IpFZFT++004/OJjbRi2TRzAewTjsZtrAYuFlKp/UMSnJNDA5csU8LCILxQ2wCDtcPGQF5iIYUT14uHeURHkEAk0F8k5aGpYJjyWytNwb5YoNfBT70bo8FIXn16LjlBiew15+zaNAwu51dBsFYqicdtkttSIaZFZ145aPlvfIh5WI8x30Oz0Cpce9I7OKGhRKF7FhGl+WPsnIinKSAiSdTkesmT5AwnXD6TQQtm35xhEUnyooXENOIkVFNER5opzVyG4lJaD1eKWnv1FVpKMoPfpJ87BU6xuF2V+B5ljnziMcP1s4os4SRrdCd/FA8PyrXiF3XzgPC8qu/fVNlOe4acd3r5ONl4d4BhnyVQRyigGS0PkdkfomAJlGYRMHPEjWyFPdUuEQtUx8RFKMV0zw/HQJ/Rw+atYkyDNY+4HfJfC1bTKUH+hg+KUteFA6JG/W0gMZiqrqDTf5m5iGw/PeThVDmBUk2PfKJypOWr1sDphndCjxCg8loE/Jqfr5kswTzycj1lDTFqZjcG8+DTjn5+4RH58BPmo3qDhtJZaB7NFPPPcX93IbGWKPcZCnepB4YBT5W5qvHRp1s1O20nLl27bCHsH6S4O15EEWwejwHmETqVtH8GTYuxO9t6PaQKQmWqu0XTH8vmTWyMX38HjVe3LTAG4vl9E6Wcszb3gASkgQiexR9hF0GUI690OmkLKtIbNHtYt1s0ebTqF89VRuGDcKdbDdbbyLXUL4Bz7GR8aaGVPn3LenwhKB8M1fTnyFl4PGsz6B2WHAyRJU6mRzvUA1dZMWLEadlasJDyg8syeW0Zq6W/TVOUntpvk3zULVS3lA9eMHB58Yk7lRnp7AcHmafWzfPve3YKy3+bKIvD5cuT08R0f1+WTcWeGvRvoH/XixkG593Oe7w2Tg1j9yfRURuw5TKw7Vae8Ymxfia4sQrW26I4roM2Pi9EO3xt3ilK8GD8cYf4R3cHSUi9U1rNCJPLr1lkmiVktYv2BR5elNxrK7aCUv/cvdeyY4LGe34iGaSXkBQ7ZFDKd21IUVq9uC0KyH00vZxWF53i34UtAiDJJojhb0FNgPQ8fKXjT9lsguIpulKANEr3BYqk8ow+NraQDFLPM1lUZbCxUeFYjF9k0nmAnbms657Ryh8nlBBVejUo906M6UVa6EahAdK2EhDn2zq4OFI4OtHJF4ROI+kFhYo5VsRTw6pMohoVh07+0iEksu/V3RVk6MIyMtydpWBtt9XQ8V2AxelcyYR6pIaW7uxWB5pgEiuhZqbz5sdq/f/NxucHd2NbjbW63r0gx1jf1IcVehsIgWd/iQQQQ8Wmp+nZz0cWzMTQX73HPnDjc935vx8Pw3gZT+ke+AlQs8mmBMd0lt3txqlVPZPySIU+gz0zU6Y5a8K9rFb98tvn0oj9bAapwUFSllF4jqDnYenlhf0oSziaN8wtTsWSCqdmJC+5NiTJ39Lw7kkGRZVgZ5GXdb4bcIo+OEoyccaQEpXFSkeRB5tR9WPfJrq35YYy9AHB6XeOlzHQlfBsyRZpyvEw+4j+xSll1icKF+9EPybV0QEXzmUc1k3t3SggdAfBEBG3GOsCQ8O6/XbCNF84xsdnmvwEvh/IS0i5UEZf3TSj1XBkXQL3pchFW7qHjJOU2S/Irk/FwZL1tN7rSmZbd8tb6wLXtmC2M+9wwj5I5zFPpvVej6faNZGXoEdfN5a0BLEeZdF6sy4Iu6oiYlgY1U7ljJ+WbHRe+Is6fF2UKduVoWO6sVkOIm9IDI415cFh3ZAaDYGPQHNqodMHUbPZZsdof7e7bzqvWkwCL+XEfpEnfP6FOT3Muz2MxfJIn8wGQoV9hPVkv/OwE6wQ6Ia5yliruqCYNIafOEbLEMHMuDIMIvE2RUbVNi79sylCqGWxVoC/BEfidE3NuQPvMqHG4rkehg3Nt8ejtINhx4WyODb8WdSU+nTv5J34E7GuuY1c0WYjqzTB7OTd8O7CM4PnVw6JtTM5iHsPBLNp22Y/WGDx/3uFXviL/PAn9NN6Mu/pS3+oaxWbNw9vNm7DL5NEjM8Z5knrHX/FawTSZH4m3gBYhmB5zVbOltpatGHr2HWH+wASpD6h70ghS5ma4vwdfjBtVoQ6pNCtAwEtnUvQKNag3byUUmbfJKLbIKSkrvGogW1GYbau4qGJ/2sGgXX1Q6WPg69XJtor2MC9eNhNS20mY/YmUV03N7NhPmnAtheVz8YnjKtinqNWbJsNO1UO3DbC0nsjWwA3kzeRhUL/YLv4sSX78CaF3UbFAzUOQtNPJWTbqVKbLsCkUp9pL3IvjgfsSdim6+IuXJqBV3F1zCjJEKP//tzHL7mK7a8T3bNHaUdvv8rcqUO7x6h0yp2Ey9VPAbFuUsSO5izQXqcbHDpY+z8Ym8ao7itBxm1/T6rnNUJKAo670D8UjkjlyIDztxVt2NyFK85v4N/nj9t9fZw3ON1LBXKEpr8oLtaqkDsc2bwHiMglU6gLk9eYZsuaokhZXnFdI9RvuBIdQvxfxvrSAe9symLc/Blgqnzuc7Bz1ijTVCx+WWCD0/dObct45Q+byg0tFIR3Yay+1vJbl7SGoazbJRC+9nBNZ6XJvSI6Z97b4bqyNBOtrZ4d1ug6E7hLOpHFPtYjxGCipKYynLVQtsdw8HQNYz1sayO8V116Xoc50ZcrYvq4eFX93S4i+iZUChIlDVUfdUvU1xmqt/QvtraZDJi+BAsfokH6UgFh0o2x7UEYvyqVhxGZuw0mv59WTJSgC+EMhh146ix0Iz0DneStHT7sekvPm6wyARi46p9ULqcVFKdf1U8ZpEY86L9yOW3SBwAv/cd435uSdMzztC6QilKpQqTapp8xDDGdqxqJCdJsya0K/mbbBtK/UR358PvmWNhw+PYhDMi6YPBvWLYUhvX/+fCNTNo3pOzltSW/L4dU3tDJpuZ/0n/q0AE0Arm7A/609Bei70oX4L9YpDgjkdBD+2WnVf6AmQMXIclEoMmoZ/gCVkX5kXl79VnpYRy/P53DagVWvqc3tqGPzxkh50xqhItzElY6hnZI02oqqsfaNZoC2g84+jpvFd+PBoS+cueC+uhMZJXudFP1fb04JEfP9xzrcpj+UdA2BjFzky0ed07rkiZyU0KlISBUsSR17/PODghMLM8Eyk07y5hRP1bfqPvQPHbT0Ur9xLrflH93Ls48cPLdo4qo8VSuZUpfqjzf9PIsU44loienQksmuRFx2WVUdOt/v/CDvh5y/4mu72S+mnggc64upeoteu+bUojq69JBkIjFxdwFSn6vUGzmAGhjEqVm27u2XoIPypDOO6OXlkLR2G5SN1SciVlFrn1j3tqdV5zBbKmM0cbp27bujwuRUe4fHpwwNYU7nQhcacuVne6KKy5X2uMm3Mzswct8c5ArADgEOaHLXk3Q9T7F4ZqWondp3BNguyPc1SGO3OJrd9MY+4qTHMfHXV6AFf09j9zNMAuNv1uPQZx7PVh9jSR/hL918P2sFDujdtZX9d6a48cMrdqNAPqK1VmwH2uovTXdgp640+MmLdDc+NYBq6vmmAuELz/IiazxY1IznC/ixvm5gzXUht0WJm1m4H1Hsndy2qBvFo3Cb2CNhPALDlIHWsskPisvQYGkaoiW62JuqhtvWAA8WnQqZI1kuhTnMx6SWaJxYYiE7ZZb8VaRpl7G4hxPIhEG2xWjZ4t0ioqVQrZ9RAcUuKr2R8iA5zYZNgRkkdO3q7XeeUXfGG6HZrnH1MIxD/CXOmAS/OSFV73PhHrL3mTHBrapzPAmM+923/CJjPETBkN8c72d3EesdO5zWTqtPOWtrvasa7MqgKWe1GLbVHfB7xuQ98/jwETW1Pplg+I1fcJwIoaQp8vcacnK8CvlIRkYB133H/JqpHfeq5KaCqywpEoFC0WoV0Gzsf5/VSuGCmMXh+1+rWhT7De7u/cdBZlmkoT+m5PsWq2AoLrvY4SQ+vo44ReOeOHc4903Ackx9h8OnA4MOcbnlUeV9h+bibHrvg5NHnn0eU/MpRsn1ZOSROOpd56CVtZdNojW6vPEYz8oKntyIjTy5yspnZZN5+XMT2bU2VJnGAwQ+bZR6tlw+k0fZrwO7Ly7YaBcVZfStYuyKXMbenvXoazVowkmHLwShuMz7miqPt+OE8EKFlusHU9GaHj/F/hM6vGTpNB8zLuLx5QbQa+VntWoJWaa9t1KAD0XERBY6gPIJyAJTs8Zgcv3M7LCgru2TYH8Mo3uaYBrpIeFL4n0UKVdrlKRXyknu2iNb1sBKVzbMk+R2/kSpN2zGt3eqQm1pByDRZKO7wEpeqTyqMJIKwiIrcXOL9OhUZXipdcXRegz770r5dSrXtV9boeCVr148DXf+I/rWmts7rBA1hA/ES6OVQZiNWTXcmZlbgnNvnbjCfze0jFj4xLGhKg3mg2q3pVFBvVSqojZflUb4pPEav0cjFbvkS07snKbX4+6wgRI6clOldhaEbtWoeQXkE5ZODcvu6+XhYft12ZW9lSf4uWcTIkqQAw3WaJCvgCV7tNA1jwsr7HbW0z9trn5eVq56CVASZSFRUwVGJKnx75gh+bsymRhhanzZjsNzfRIr6MH7dqUCt9ts3P776/urVj29//uNfL7CY7spTD2NkXocDCK64TFztXYI3kTKZ49uu5PimDFI0aHyqfsh0qYYKFKfvtzQLFKw67YjcjHxiOb8p/Ol72Q0zGP3zkj2zDGMyOh3Bnjn4r3/9a32fLxIYF43oykOt3wA85am4eQ8/9OMXzLqMvRU+cqH/9HOVe5USZwxeX8aYazf/w8rLn1/GQH5MCP0wcD3hc5sH9jQIzb3gQ1nv4OPJ7qQYgP06MW6mYgcEdL++3HxFdAMiPjx39C2nHzjwZckET5VbTiaWS0STF/n3Pho5g00q0QVrmwzUrocyLr783nmcsCxZ3oquSIDF4voO2HUjh3MjssHskF//8HddasAr/od3X+tSzjbHKVXO7adGJYj5IriMp7sWnO1aUM6vjUkbZDc2Bvu+ZVfB9+/Yn5Q6k0YcWgaG+Ty9ognuJeEMUE9h5vWzOT4BpfRGwLJEsIffC7Fc4wVN/Hvcxx96BrccPhXBzDo/nzt7+fh/otEhQDe5urxVVRRVksDDsgGJ1mFS3m1FGpdx8VvTLJr8ilXaOmO6mTOGDSBpOZ3UCXTMu9tmhf0zX80KMF6Ypqq2Mm3/8/sSIaAqTb4OUQ4iQp2Hy5wPoNgn1wKvgDelNkJE+lkpJ/1ECUvO14OC+IrViH3FCjKlSEoy5UT/ipYg6LwJa8jYj38XEQUiRAdIcXWXJvH1LnL6NkkbM5e4vlahKik5CPqbRDFzaMhZ3xXmRnU0cILSsxS0sXWk3vR9R7xVuSiPUf7nrimmjn0eOo5wXSEejVTZreFBwEbQVnpyJcbNE/AMVYMZZcN6AeRestnELV+UcXZK3QYLjPry985PzSGyeWDiTjU+4kyG2+GHcGcnNDXrQOPUCE0Xdp2NVSY26mEnO+TYyd7t3+3eGfw7NbPSjaJYRa+t6xb/JUf033Lx6WPSPagnFMuTBkuJB2qZ39AtDxNOw7hjsV6Iu8s43OSbVFzJmpim/WSMeu9MXYuHlml7M8c1pvuA2t7GVpzAAAxX+vBoRql/Gl8csUA1gC4YwASG4LLMiZohbuV2PV9UQgLoy9XYBKY5r98uxx1Ic1Fr85rZenWpvivXFzIAEHcAw8kGhggDh26N2yDsW0qY36KtC8joZFI/aythtJBJJlU0sWwRLZMF8MJxLmNQd0BwyJgulFdikcAXTHy6jDUrXzJF6YxJKl0U+DUFRquiXxa+wjc4l5xIuldlW5IOiUgpehlASQutrN6hoe2g/e9dMMvkPkB5BBu8OmG5VrkS4YzHcK+BWCUcLzBgapzp8vXieGaqwizgxlXFWUBN7pklCyT/kYbjwuBFachkF1YcN0sYV67WPIVWt8YoB5YdBqEx98V8NjUs6Vj46D2EzgeGy/su/VX2tIgWjOIcINBnyjuY9Qte825Ng8JE3MsjZD017UtGp8Wk2UkEBzoyP/m+ZfJz10l5B4cqZ/m1SCwdfO499q8yr6p3dMxnmwwb6JvQFMlWi+pf2MLIN1fwgP6PLzF8p3UZQyeaj5sTVk0+VCOj4oTLl6yL+AvWoPuQOekgH1zdtAp6ZOTjiktWWhd2W88oI+yzmTth7eIxj0AfTnNl1MXidqW0nn1Ix6MlErMfUqxLd8xMEwTCmTrzmTWzwsBw9mOqbCrzoEajYtw90Cfki9YtWpuNkXuNvTOtaxPQTqmoOVWk+aQr+zyL/BYT9YS7FkH53DF0TqwGfWU+zjYr1RTqdXeJPBLM9GkBbSBV1spCCjqIKmoYwEzg3LVIocof8aH8xfgyS6gh7Bt93VhdHlO0e9IC8Pavd+/y+AtPKcFMmiZ3ANxn9unUNWiJCaNblTYzw1snUKqekBX4YBpfwthUagHQwte0XslgTXoRFqBa30s6aumrl0cmSeM4QNuGxkd80j7wwnXD6TQQtm35xp64gb2jLKIDY6FcaisBXxZ9gMWAokCOwigHVpghKpuMQe8x1ZqiU1/hiPXAd/YHMptV/vd83Jd9IN7JTzHNEsnGpwSWnAqLFrtysateAEsrnZD1SHQ71ymPYMu5FQ8kR37OexdCPfbdTZTnqLrgsF4nGy8PI5ggQ76Klhjd7Q7Ye0sTptwQnzDTInvs+WTIYIXvR3ykpi1Mx+DefBpwzs/d38wY62uoPMXvfPXAag97VWts1Bd/EEFU2f8Tz/3FPUxpOX23mACE1lsZDL1LanR4v5Qv36XR+0jUaoOoVMF2Sq4OAChn3hYK7mRYCAmDAgJWGQa1fthU2SFELZfcutXOmo/84HeRwfgjgcq+ZQl9jdZJcURiTByj8hQm0DiP8nv6BPAUJg7udVF7Yg0UHXc0aM2MqXPu21MBWwzhm3vcZXRqjKbBdMflHhwe4OAiCmim/L5gCofHJ6w9xK/ki5Idcn5XZ5HK1PcUXG4cZnX1tKTWcXA1bp+/byGVBzNVZb5XMKgMLKIVfVff6LPAE9MouFQ8toG/J8jkyqsOnRoJDiFBNSgXayAzdrneO8N+1+uNcM1hzonpRMRbJkmgpjdAcbbg6Q1onXfRijzPdKJGwWmOQ5srFLjSBF4yB6GN4Gq+MJtHmtuc+GzfdIKZsK3pnNvOnhfyyoj1n7uOubLzxjHp+vIjpouX2m2oNO03DT6juFZzBOrWHyrdGOlxdzgud0MsBbUaVstgFLs9MRQ6uXL/RZJqmel047hLojJSWMj21iuSxmUMH7KiVn1jlozu8M9syEXeV+oTzS7OfHsWzttFJJZc7RtNtUHSZgYcgMt8gZZddGWQExhn9pnyJoszmPqWKxHDevWKWV+WGyl9huYDl65BQAXZDWgWuLUqm6XHkiCWsvUOZ8wxoX3uuXOHm57vzXh4/imzRX/1xZEz7k17Cld8U2kU8tQR/xx5wLd//v4+K0ZK29pn0L0duCmrafbxGnNbzKy00cugV7+ZQmPnir0LrXY68bo8btXx3KQNU6ubyoAB+xw8jwgwelz5zO3ycZOKlIwzRzS0MirqlWv0v2LVt5LsTz2nqW1qI/yN1c1Ey57ZwpjPPcMIuePsyR+9OHnGTuvodzp/BnQY1zWZU0bmEMiYO2FvQlVUe/AvZS3LYmcM49SZ1qmMZ1dShJ1nMzYepbtkdUWC3JvvlD9A+2jqzT4kvh9pj5Ds1qvQ+5Ytan5qP1LZyVFkD3UYWPiV9vO44dCw4MGVlJviM41ZP3EHv6oah6F0SeuMVegMep/KAhXvUdXyHGhUqA99ho3KIz0bDiSrfp8nSreuWXdNN5wtAGKRdn3YYwpTyNeuAujKTfB6z8dMQ4/Ovj6spGf+IkmWyg6WtVNHdeQpUYqRTE7foKF8qFte9sl9s2h7y9T4cqp8bvtiDwhpJxHU/KJae6px8YT2LqPaotvAJ1kaJNbgjR6m/HFNXgEz9QNK037G7lqCm1TpCmdJ9hr3n5LkGf3rdRGp8u0ypl9Xiouqdgb8r1QvJ4p6VSmbyy9qNk+MJVCl+bDp44DfT2eO9Eo+b5kJiQ1vFCl00gNdRxoctwoPwGGuDc84j08zvYP3QpUt5U2r6mWsCbP0ix3ZvXVGKU5F9Vnaw/lJV4wX5JyFUyY2AP+mm/gUc5XTbk05W1rqRnT3IPeeoX60ZFY6mZ7iMq0CZIWRcC3eK7aojW2pR3YkeUfrTv8XATMKuq273biXF9z8FE94dvX9VPuSK90B8vsslY/ysdW1hGudBTtbKduuf8YaDQ27dOpSD5q39i7rthOszrOrN9OYs/cy1luMK5n+F3NRX8a6cEXzd7s9a4tbwlS9rAn91gQbLZyxBvVhrjYJPn/StOhdd0i72VpLg9wX6m/HjNqTCjXp8zOb6pTdhVtSkxZ5nOpSp8ooHdfusTaPHCuf2igYPFDIPQIdVM1cUw9p8sRJxvsvsYnOO2wYf7CRlR6e1nJpB7eBWrXKMpKnrJP/XRJoFDwpKFWIVuTyUqbM7vzkhr9lPY9X5gqpTGvSzdaaSnfH5/oU2bkH9bdHZpmVuVIp8+jw1SvLnjxpHukdB71InmDgOwSpUKUeo7vppbkYHGx9TK3S1R5Sr+uqnX7/AFvSgWW1j1zIOwvLEzpl73aDVrlFaWtprzu1NNVZKRdp8HxJkmhvLKu3ZvBvKKhqn0lT6bD5F6s8SOvauzD/LKR7UTvxaFeeUdKpd80b+j+XMZB9QeFxnapUyzNI5kyeND1u1wUZ2Zm8M9iGA3ujpLkzUuYluayK2wgaQ/Gukzjr3Sq+6yHfdIcVcvYZedlOGqYxjS/gFZZhKRi6GiEf4kax+FGWfsHqJTU81c8td/Ocyd6T+j5iHirBWCTTqWU3bt9l6s/RI2/ydqfxfaO31DBsum7U8/mUeXvVx1P7oMhnWlVGp/uB1MyThvtV9xzXEQ1C0ZSQoB68VO28KAwH3vMnz467s99PkQmxcs1IPqlcJsIkhDjFnLDp1BltAmwSLDVCbeoba/A7dPLYnWIESeMV5o7MEpWFvh20op/RXbfakc/w3VhlZqrinjE8Bt5PMPMytFwNMyXhqv+QNibV2+rstVWqW6X2OJkdPuNqh9TkJ0z3iCOVOgVzeHYGEWhmE21E9mUDamk5U7QWjTbbLcwCSik967LAx9jSVRJeUX7Ql1IujHUQaT16UatLTZC4mgUfphDtWzbbXAvkDFqq88Usa087o+XAzpJ2qKqa+l1WU7nchvXFeq3R087jE751AFiNoDpIdUAHQ1JGWXULps675gm/TsoINewpMoOMr1aHy2Jzq9UVAenXLLJem5E1eeLUkp3eyn1xDiifZJlaWN1DvCvdOepZDLUSozw+FirLoUpkqDMyWvpaotKC7GlH2kYbWyN3EtVeb+FGY7N6YwPeqbPJk2dPHNZjZc4rVP7LnFmUZGybEjuYLKtVnNJh9X9jW91CykMo44UzdkJ6fD6sgZ1cM0oa9bA85NJxIcZwj4h0BDQocoq184n139IZlDD1Vc8TvcLo0Pbb8tjBrKVD7UyeON3eLl8AHdI96BNQl59kYrvtRbZIt1nU0Rqu2qK3hQP9kofi23fu6HqguoSSkw9ku2Q0ke9IQyIuSK8FVeMFK0qXO/da+edPnqdu14VlOOBE9ykqXxGb8CbCv2Wmpp4D0w7rHF+uODtTeZaqZzudhrhDp3Hrx38jM1TemZpLXqtQmadMFdn3TazYecruk00NmhW+9x+o/SJc32GOMo39J08bKx3AMFo9s2hFi4To4e4gS5uJxICzINor+FHjrnpgDfvyVRGCinOpb2te4xNF67uS8BnTjQ4fYl92Wfm3NUoWXdxv67afHyCf2V42c+XAulLbgMoPT6/UU5xM+/zEe+jUMtlcxlctcifwcrS/96Mz0Qxbv3cZy6lO11MzEa260xV1sRbXRkVLn/S0rBZZHK3XIh9w+v7VS/Lwqa7aYaVUGqetEpAe+73aaa2OdNE4kS78hjGZ9EcRHQLNSKwQRoYPAs0x0ecP+P20M3DQ7FhJwyHtIZj5Yz/LdKHnK6p6xn6gRfogeTp24oq+pWpWmHOKBx8HZ88uSoc5MbqDrA7JGSpNnjh5z86Rq1V00I4vdy2ag5mwtxFeASqy0NAZV5GJptz6aFcxTLiDnmFQZuc2SufKuzTKH3CMT70hSePfeLhFD6pSX4sHGZVHiuWf/w+u/aiT')
assert len(COHORT)==48 and len(RAW)==144
assert len({r['problem_id'] for r in RAW})==24
assert len({(r['model'],r['problem_id'],r['phase']) for r in RAW})==144
CASE={c['id']:c for c in MANIFEST['cases']}
assert {r['problem_id'] for r in RAW}.issubset(CASE)
# Stable pseudonyms and random ordering. The review file hides model, phase,
# stored label and ground truth. Human reviewers must assess the entire
# numerical answer, not merely match a regular expression.
public=[];truth=[]
for record in RAW:
    seedkey=record['model']+'|'+record['problem_id']+'|'+record['phase']
    key=hashlib.sha256(('blind-audit-20261025|'+seedkey).encode()).hexdigest()[:20]
    c=CASE[record['problem_id']]
    public.append({'response_id':key,'question':c['prompt'],'model_response':record['output'],
                   'review_label':'','failure_reason':'','reviewer_comment':''})
    truth.append({'response_id':key,'model':record['model'],'problem_id':record['problem_id'],
                  'dataset':c['dataset'],'phase':record['phase'],
                  'gold_answer':c['target'],'stored_strict_label':record['stored_label']})
public.sort(key=lambda r:r['response_id']);truth.sort(key=lambda r:r['response_id'])
pub=pd.DataFrame(public);key=pd.DataFrame(truth)
for who in ['reviewer_A','reviewer_B']:
    path=OUT/(who+'_BLINDED.csv')
    if not path.exists():pub.to_csv(path,index=False)
key.to_csv(OUT/'SEALED_KEY_DO_NOT_SHARE_WITH_REVIEWERS.csv',index=False)
protocol={'study':'RSX_R05','review_items':len(public),'source_manifest_hash':digest(MANIFEST),
          'review_sample_rule':'12 lexicographically lowest hashed case IDs per benchmark; identical questions across models',
          'rubric_version':'semantic_rubric_v1','labels':['correct','wrong','ambiguous'],
          'reason_codes':['explicit_correct','explicit_wrong','incomplete','contradictory','unclear','other']}
(OUT/'FROZEN_BLINDED_REVIEW_PROTOCOL.json').write_text(json.dumps(protocol,indent=2))
print('Blinded review files created:',len(pub),'answers; gold/model/phase in separate sealed key.')
print('Blank labels are intentional. Do not publish semantic accuracy before independent review.')


### Reviewer instructions

Two independent reviewers should receive **only** `reviewer_A_BLINDED.csv` and `reviewer_B_BLINDED.csv` respectively, never the `SEALED_KEY...` file. For each output decide whether the model's *final intended numerical answer*, including a clear unmarked prose conclusion, is mathematically correct; explicitly incorrect; or missing/incomplete/contradictory. A numerical value mentioned only in working is **not** necessarily the final answer. Use correct / wrong / ambiguous exactly, with a reason code. Do not assign labels by matching gold numeric text alone. Reviewers should not communicate before submitting. Disagreements need an independent adjudicator; the notebook never resolves them automatically.

In [ ]:
# Stage 2 — Import independent reviewer files and finalize only when adjudicated.
# In Colab: upload edited reviewer CSVs to /content, then run this cell.
import shutil
for who in ['reviewer_A','reviewer_B']:
    local=Path('/content')/(who+'_BLINDED.csv')
    if local.exists():shutil.copy2(local,OUT/local.name)
A=pd.read_csv(OUT/'reviewer_A_BLINDED.csv').fillna('')
B=pd.read_csv(OUT/'reviewer_B_BLINDED.csv').fillna('')
allowed={'correct','wrong','ambiguous'}
assert A['response_id'].is_unique and B['response_id'].is_unique
assert set(A['response_id'])==set(pub['response_id'])==set(B['response_id'])
complete_A_B=(set(A['review_label']).issubset(allowed) and set(B['review_label']).issubset(allowed))
if not complete_A_B:
    print('PENDING HUMAN ANNOTATION. Each reviewer must label all responses independently.')
else:
    from sklearn.metrics import cohen_kappa_score
    merged=A[['response_id','review_label','failure_reason']].merge(B[['response_id','review_label','failure_reason']],on='response_id',suffixes=('_A','_B'),validate='one_to_one')
    merged['agrees']=merged['review_label_A']==merged['review_label_B']
    merged.to_csv(OUT/'inter_rater_raw_agreement.csv',index=False)
    kappa=float(cohen_kappa_score(merged['review_label_A'],merged['review_label_B'],labels=sorted(allowed)))
    if math.isnan(kappa):
        print('Cohen kappa not estimable: all annotated responses occupy one category.')
        kappa=None
    print('Agreement:',float(merged['agrees'].mean()),'Cohen kappa:',kappa,'disagreements:',int((~merged['agrees']).sum()))
    disagreements=merged[~merged['agrees']]
    disagreements.to_csv(OUT/'THIRD_REVIEWER_ADJUDICATION_REQUIRED.csv',index=False)
    merged['final_label']=merged['review_label_A']
    fully_adjudicated=True
    if len(disagreements):
        third_path=OUT/'THIRD_REVIEWER_BLINDED.csv'
        if not third_path.exists():
            third=disagreements[['response_id']].merge(pub,on='response_id',validate='one_to_one')
            third.to_csv(third_path,index=False)
        uploaded=Path('/content/THIRD_REVIEWER_BLINDED.csv')
        if uploaded.exists():shutil.copy2(uploaded,third_path)
        third=pd.read_csv(third_path).fillna('')
        assert third['response_id'].is_unique
        assert set(third['response_id'])==set(disagreements['response_id'])
        if set(third['review_label']).issubset(allowed):
            tmap=third.set_index('response_id')['review_label'].to_dict()
            merged.loc[~merged['agrees'],'final_label']=merged.loc[~merged['agrees'],'response_id'].map(tmap)
        else:
            fully_adjudicated=False
            print('THIRD REVIEWER PENDING. Give only THIRD_REVIEWER_BLINDED.csv to an independent reviewer.')
    if fully_adjudicated:
        result=merged.merge(key,on='response_id',validate='one_to_one')
        result['semantic_correct']=result['final_label']=='correct'
        pivot=result.pivot(index=['model','problem_id','dataset'],columns='phase',values='semantic_correct')
        assert pivot.notna().all().all() and len(pivot)==48
        for model in ['model_A','model_B']:
            selection=pivot.loc[model]
            for action in ['finalize','regenerate']:
                wins=int(((~selection['source']) & selection[action]).sum())
                damaged=int((selection['source'] & (~selection[action])).sum())
                print(model,action,'n:',len(selection),'recovered:',wins,'damaged:',damaged,'paired exact p:',float(binomtest(min(wins,damaged),wins+damaged,0.5).pvalue) if wins+damaged else 1.0)
        result.to_csv(OUT/'HUMAN_ADJUDICATED_OUTCOMES.csv',index=False)
        (OUT/'R05_ADJUDICATION_COMPLETE.json').write_text(json.dumps({'records':len(result),'kappa':kappa,'disagreements':int((~merged['agrees']).sum()),'resolved':True},indent=2))
        print('HUMAN ADJUDICATION COMPLETE; all 144 response labels are resolved.')


In [ ]:
zip_path=Path('/content/RSX_R05_Blinded_Grading_Package.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for file in OUT.iterdir():
        if file.is_file():z.write(file,file.name)
print('Package:',zip_path)
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError: pass
